In [ ]:
# caption生成 图->文

import base64
from openai import OpenAI
from dotenv import load_dotenv

# 初始化客户端
load_dotenv() 
client = OpenAI(
    base_url="https://www.autodl.art/api/v1",
    api_key=os.environ.get("AUTODL_API_KEY"),
)

# 读取本地图片并转换为base64
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 图片路径（请替换为您的实际图片路径）
image_path = "backend/output/panel4.jpg"
base64_image = encode_image_to_base64(image_path)

# 调用接口（不使用流式响应，加入enable_thinking参数）
completion = client.chat.completions.create(
    model="Qwen3.5-397B-A17B",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                },
                {
                    "type": "text",
                    "text": "Describe this image in a single prose paragraph. For each character, start by clearly stating their relative position (e.g., 'the character on the left', 'in the foreground', 'the girl on the right'), then describe their appearance (hair, clothing), and finally their actions or emotions. Do not use specific names. Ignore all embedded text, speech bubbles, and dialogue. Focus purely on visual elements."
                }
            ]
        }
    ],
    extra_body={
        "enable_thinking": False
    }
)

print(completion.choices[0].message.content)

In [ ]:
# prose生成 文->文

from openai import OpenAI
from dotenv import load_dotenv
import os

# 初始化客户端
load_dotenv() 
client = OpenAI(
    base_url="https://www.autodl.art/api/v1",
    api_key=os.environ.get("AUTODL_API_KEY"),
)


# 调用接口
completion = client.chat.completions.create(
    model="Qwen3.5-397B-A17B",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "总结prose内容"
                }
            ]
        }
    ],
    extra_body={
        "enable_thinking": False
    }
)

print(completion.choices[0].message.content)

In [4]:
# 人物参考图生成 图文->图

import os
import base64
import mimetypes
import urllib.request
from io import BytesIO
from PIL import Image
import dashscope
from dashscope import MultiModalConversation
from dotenv import load_dotenv
from dashscope.aigc.image_generation import ImageGeneration
from dashscope.api_entities.dashscope_response import Message

# 以下为北京地域base_url，各地域的base_url不同
dashscope.base_http_api_url = "https://dashscope.aliyuncs.com/api/v1"

# 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")


# --- Base64编码函数 ---
# base64编码格式为 data:{MIME_type};base64,{base64_data}
def encode_file(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)
    if not mime_type or not mime_type.startswith("image/"):
        raise ValueError("不支持或无法识别的图像格式")
    with open(file_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_string}"


def pad_image_to_min_size(image_path, min_width=240, min_height=240):
    """
    将图片放置到最小尺寸的空白画布上，不缩放、不裁剪，保持原图分辨率。
    画布颜色为白色（可改为透明）。
    """
    img = Image.open(image_path).convert("RGBA")  # 转为RGBA以支持透明
    w, h = img.size

    # 如果尺寸已经满足要求，直接返回原图
    if w >= min_width and h >= min_height:
        return img

    # 创建白色背景画布（如果要透明背景，把 (255,255,255,255) 改为 (0,0,0,0)）
    canvas = Image.new(
        "RGBA", (max(w, min_width), max(h, min_height)), (255, 255, 255, 255)
    )
    # 居中粘贴原图
    x = (canvas.width - w) // 2
    y = (canvas.height - h) // 2
    canvas.paste(img, (x, y), img)  # 使用原图自身的alpha通道作为mask
    return canvas


def encode_file(file_path):
    """
    预处理图片并返回Base64编码字符串。
    会自动补齐小图到240x240。
    """
    # 先补齐尺寸
    # wan2.7
    # padded_img = pad_image_to_min_size(file_path, 240, 240)
    # qwen-image
    padded_img = pad_image_to_min_size(file_path, 512, 512)

    file_name = os.path.basename(file_path)
    padded_img.save(f"output/padded_{file_name}")

    # 将PIL Image转为字节流，并获取MIME类型
    buffer = BytesIO()
    padded_img.save(buffer, format="PNG")  # 统一保存为PNG保证质量
    buffer.seek(0)

    mime_type = "image/png"
    encoded_string = base64.b64encode(buffer.read()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_string}"


"""
图像输入方式说明：
以下提供了三种图片输入方式，三选一即可
1. 使用公网URL - 适合已有公开可访问的图片
2. 使用本地文件 - 适合本地开发测试
3. 使用Base64编码 - 适合私有图片或需要加密传输的场景
"""

image_1 = encode_file("/home/zaln/下载/111/char1.png")
image_2 = encode_file("/home/zaln/下载/111/char2.png")
image_3 = encode_file("/home/zaln/下载/111/char3.png")
image_4 = encode_file("/home/zaln/下载/111/char4.png")
image_5 = encode_file("/home/zaln/下载/111/char5.png")


# message = Message(
#     role="user",
#     content=[
#         {"text": "根据参考图的美术风格生成角色设定集，完整复刻参考图人物面部、肤色、发质、体型、服装。核心任务：展示标准三视图，正面、侧面、背面全身站姿，比例一致，参考图的服装褶皱契合，纯白色背景+细微物理阴影。"},
#         {"image": image_1},
#         {"image": image_2},
#         {"image": image_3},
#         {"image": image_4},
#         {"image": image_5},
#     ],
# )
# print("----sync call, please wait a moment----")
# rsp = ImageGeneration.call(
#     model="wan2.7-image-pro",
#     api_key=api_key,
#     messages=[message],
#     watermark=False,
#     n=4,
#     size="2K",
# )

# # 提取结果图片URL并保存到本地
# if rsp.status_code == 200:
#     os.makedirs("output", exist_ok=True)
#     for i, choice in enumerate(rsp.output.choices):
#         for j, content in enumerate(choice["message"]["content"]):
#             if content.get("type") == "image":
#                 image_url = content["image"]
#                 file_name = f"output/ref_{i}_{j}.png"
#                 # 结果URL有效期为24小时，请及时下载
#                 urllib.request.urlretrieve(image_url, file_name)
#                 print(f"Image saved to {file_name}")
# else:
#     print(f"Failed: status_code={rsp.status_code}, message={rsp.message}")



messages = [{
    "role": "user",
    "content": [
        {"text": "把图2的涂鸦喷绘在图1的汽车上"},
        {"image": image_1},  # base64 编码后的图片
        {"image": image_2},
        {"image": image_3},
        # {"image": image_4},
        # {"image": image_5},
    ]
}]

# 调用
response = MultiModalConversation.call(
    api_key=api_key,
    model="qwen-image-2.0",
    messages=messages,
    stream=False,
    n=4,
    watermark=False,
    negative_prompt=" ",
    prompt_extend=True,
    size="2048*2048",
)

if response.status_code == 200:
    os.makedirs("output", exist_ok=True)
    
    for i, choice in enumerate(response.output.choices):
        for j, item in enumerate(choice["message"]["content"]):
            if "image" in item:
                image_url = item["image"]
                file_name = f"output/ref_{i}_{j}.png"
                urllib.request.urlretrieve(image_url, file_name)
                print(f"Image saved to {file_name}")
                
    # 打印生成信息
    usage = response.usage if isinstance(response.usage, dict) else {}
    print(f"Generated {usage.get('image_count', 1)} image(s) "
          f"at {usage.get('width', 2048)}x{usage.get('height', 2048)}")
else:
    print(f"Failed: {response.status_code}, {response.code}, {response.message}")

Image saved to output/ref_0_0.png
Image saved to output/ref_0_1.png
Image saved to output/ref_0_2.png
Image saved to output/ref_0_3.png
Generated 4 image(s) at 2048x2048


In [5]:
# 视频生成流程 参考图+文本 -> 视频

import requests
import time
from dotenv import load_dotenv

load_dotenv()

# ========== 配置 ==========
SUBMIT_URL = "https://www.autodl.art/api/v1/vidu/ent/v2/reference2video"
API_TOKEN = os.getenv("AUTODL_API_KEY")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Token {API_TOKEN}"
}

def image_to_data_uri(file_path):
    """
    将本地图片文件转换为 data URI 字符串
    """
    # 根据扩展名判断 MIME 类型
    ext = os.path.splitext(file_path)[1].lower()
    mime_types = {
        '.png': 'image/png',
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.webp': 'image/webp',
    }
    mime = mime_types.get(ext, 'image/png')  # 默认按 png 处理
    
    with open(file_path, 'rb') as f:
        image_data = f.read()
    
    # 检查解码后大小是否超过 10MB
    if len(image_data) > 10 * 1024 * 1024:
        raise ValueError(f"图片 {file_path} 大小超过 10MB，无法使用")
    
    b64 = base64.b64encode(image_data).decode('utf-8')
    return f"data:{mime};base64,{b64}"

# 使用示例
local_images = [
    "/home/zaln/下载/111/ref_1_0.png"
]

data_uris = [image_to_data_uri(img) for img in local_images]

# 任务参数（与之前一致）
payload = {
    "model": "viduq2",
    "images": data_uris,
    "prompt": "角色在古老图书馆中跳芭蕾舞，烛光摇曳，气氛诡异。",
    "duration": 2,
    "aspect_ratio": "16:9",
    "resolution": "540p",
    "bgm": True,
    "watermark": False,
    "off_peak": False
}

# ========== 1. 提交任务 ==========
try:
    submit_resp = requests.post(SUBMIT_URL, headers=headers, json=payload)
    submit_resp.raise_for_status()
    task = submit_resp.json()
    task_id = task.get("task_id")
    if not task_id:
        print("未获取到 task_id，响应内容：", task)
        exit()
    print(f"任务提交成功，ID: {task_id}")
except requests.exceptions.RequestException as e:
    print(f"提交失败: {e}")

提交失败: 401 Client Error: Unauthorized for url: https://www.autodl.art/api/v1/vidu/ent/v2/reference2video
